# Experiment: GP-KGE (Ours)

**Loss:** BCE
**Kernel:** Relation-Aware

In [ ]:
# Setup
import os, sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !rm -rf /content/kg-bayesian-prior
    !git clone https://github.com/ChorokLeeDev/kg-bayesian-prior.git /content/kg-bayesian-prior
    !pip install -q torch-geometric gpytorch networkx pandas tqdm scikit-learn
    os.chdir('/content/kg-bayesian-prior')
    sys.path.insert(0, '/content/kg-bayesian-prior')
else:
    sys.path.insert(0, os.path.dirname(os.getcwd()))

In [ ]:
import gc, json, warnings
import torch
import torch.nn.functional as F
import numpy as np
from scipy import sparse
from scipy.sparse.linalg import eigsh
from tqdm.notebook import tqdm

from src.data import load_fb15k237
from src.models import GPKGE
from src.kernels.matern_graph import GraphLaplacian
from src.utils.training import set_seed
from src.evaluation.calibration import expected_calibration_error, brier_score
from src.evaluation.ood_detection import compute_auroc, create_ood_dataset

warnings.filterwarnings('ignore')
set_seed(42)
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

train_data, _, test_data = load_fb15k237()
print(f"Data: {len(train_data):,} train, {len(test_data):,} test")

In [ ]:
# Model setup
print("Setting up GP-KGE...")

model = GPKGE(
    train_data.num_entities,
    train_data.num_relations,
    embedding_dim=200,
    kernel_type="relation_aware",
    num_inducing=500
).to(device)

# Eigendecomposition
print("Computing eigendecomposition...")
kernel = model.kernel
kernel.num_entities = train_data.num_entities
kernel.relation_laplacians = {}

success, failed = 0, 0
for rel_id, adj in tqdm(train_data.relation_adjacencies.items(), desc="Eigendecomp"):
    if adj.nnz < 10:
        continue
    try:
        degrees = np.array(adj.sum(axis=1)).flatten()
        D_inv_sqrt = sparse.diags(1.0 / np.sqrt(np.maximum(degrees, 1e-10)))
        L = sparse.diags(degrees) - adj
        L_norm = D_inv_sqrt @ L @ D_inv_sqrt
        L_norm = (L_norm + L_norm.T) / 2
        k = min(100, L_norm.shape[0] - 2)
        if k < 2:
            continue
        eigvals, eigvecs = eigsh(L_norm, k=k, which='SM', maxiter=1000, tol=1e-4)
        kernel.relation_laplacians[rel_id] = GraphLaplacian(adj.shape[0])
        kernel.relation_laplacians[rel_id].eigenvalues = torch.tensor(eigvals, dtype=torch.float32)
        kernel.relation_laplacians[rel_id].eigenvectors = torch.tensor(eigvecs, dtype=torch.float32)
        success += 1
    except:
        failed += 1
print(f"Eigendecomp: {success} success, {failed} failed")

In [ ]:
# Training
print("\nTraining GP-KGE...")
opt = torch.optim.Adam(model.parameters(), lr=0.001)

for ep in (pbar := tqdm(range(50), desc="GP-KGE")):
    model.train()
    loss_sum, n = 0, 0
    for st in range(0, len(train_data), 1024):
        pos = torch.tensor(train_data.triples[st:st+1024], device=device)
        neg = pos.clone()
        neg[:,2] = torch.randint(0, train_data.num_entities, (len(pos),), device=device)
        opt.zero_grad()
        ps = model.score_triple(pos[:,0], pos[:,1], pos[:,2], use_mean=True)
        ns = model.score_triple(neg[:,0], neg[:,1], neg[:,2], use_mean=True)
        loss = F.binary_cross_entropy_with_logits(
            torch.cat([ps, ns]), torch.cat([torch.ones_like(ps), torch.zeros_like(ns)]))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        loss_sum += loss.item()
        n += 1
    pbar.set_postfix(loss=f"{loss_sum/n:.4f}")

print("Training done!")

In [ ]:
# Evaluation
print("\nEvaluating...")
model.eval()

# MRR
ranks = []
with torch.no_grad():
    for i in tqdm(range(0, len(test_data), 200), desc="MRR", leave=False):
        batch = test_data.triples[i:i+200]
        h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
        scores = model.score_tails(h, r)
        target = scores[torch.arange(len(t), device=device), t]
        ranks.extend(((scores > target.unsqueeze(1)).sum(1) + 1).cpu().tolist())

ranks = torch.tensor(ranks, dtype=torch.float)
mrr = (1/ranks).mean().item()
h1 = (ranks <= 1).float().mean().item()
h10 = (ranks <= 10).float().mean().item()

# ECE
pos = test_data.triples
neg = np.array([[h, r, np.random.randint(train_data.num_entities)] for h,r,t in pos])
all_t = np.vstack([pos, neg])
labels = np.concatenate([np.ones(len(pos)), np.zeros(len(neg))])

confs = []
with torch.no_grad():
    for i in tqdm(range(0, len(all_t), 1024), desc="ECE", leave=False):
        batch = all_t[i:i+1024]
        h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
        scores = model.score_triple(h, r, t)
        confs.append(torch.sigmoid(scores).cpu().numpy())
conf = np.concatenate(confs)
ece, _ = expected_calibration_error(conf, labels)
brier = brier_score(conf, labels)

# AUROC
ood_t = create_ood_dataset(train_data, test_data, "random", len(test_data))

def get_unc(triples):
    uncs = []
    with torch.no_grad():
        for i in range(0, len(triples), 1024):
            batch = triples[i:i+1024]
            h, r, t = [torch.tensor(batch[:,j], device=device) for j in range(3)]
            pred = model.predict_with_uncertainty(h, r, t)
            uncs.append(pred['total'].cpu().numpy())
    return np.concatenate(uncs)

auroc = compute_auroc(get_unc(test_data.triples), get_unc(ood_t))

results = {"mrr": mrr, "hits@1": h1, "hits@10": h10, "ece": ece, "brier": brier, "auroc": auroc}
print(f"\n{'='*50}")
print("GP-KGE RESULTS")
print(f"{'='*50}")
print(f"MRR: {mrr:.4f}")
print(f"Hits@1: {h1:.4f}")
print(f"Hits@10: {h10:.4f}")
print(f"ECE: {ece:.4f}")
print(f"Brier: {brier:.4f}")
print(f"AUROC: {auroc:.4f}")

In [ ]:
# Save
with open('gpkge_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("Saved to gpkge_results.json")